### 加载数据集

In [1]:

from utils import load_corpus, stopwords

TRAIN_PATH = "./data/weibo2018/train.txt"
TEST_PATH = "./data/weibo2018/test.txt"

In [2]:
# 分别加载训练集和测试集
train_data = load_corpus(TRAIN_PATH)
test_data = load_corpus(TEST_PATH)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\20429\AppData\Local\Temp\jieba.cache
Loading model cost 0.499 seconds.
Prefix dict has been built successfully.


In [3]:
import pandas as pd

df_train = pd.DataFrame(train_data, columns=["words", "label"])
df_test = pd.DataFrame(test_data, columns=["words", "label"])
df_train.head()

,words,label
0,书中 自有 黄金屋 书中 自有 颜如玉 沿着 岁月 的 长河 跋涉 或是 风光旖旎 或是 姹...,1
1,这是 英超 被 黑 的 最惨 的 一次 二哈 二哈 十几年来 中国 只有 孙继海 董方卓 郑...,0
2,中国 远洋 海运 集团 副总经理 俞曾 港 月 日 在 上 表示 中央 企业 走 出去 是 ...,1
3,看 流星花园 其实 也 还好 啦 现在 的 观念 以及 时尚 眼光 都 不一样 了 或许 十...,1
4,汉武帝 的 罪己 诏 的 真实性 尽管 存在 着 争议 然而 轮台 罪己 诏 作为 中国 历...,1


### 特征编码

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(token_pattern='\[?\w+\]?', 
                             stop_words=stopwords,
                             max_features=2000)
X_train = vectorizer.fit_transform(df_train["words"])
y_train = df_train["label"]

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:3: SyntaxWarning: invalid escape sequence '\['
C:\Users\20429\AppData\Local\Temp\ipykernel_20692\978941121.py:3: SyntaxWarning: invalid escape sequence '\['
  vectorizer = CountVectorizer(token_pattern='\[?\w+\]?',
C:\Users\20429\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['元', '吨', '数', '末'] not in stop_words.
  warnings.warn(


In [5]:
X_test = vectorizer.transform(df_test["words"])
y_test = df_test["label"]

### 训练模型&测试

In [6]:
import xgboost as xgb

param = {
    'booster':'gbtree',
    'max_depth': 6, 
    'scale_pos_weight': 0.5,
    'colsample_bytree': 0.8,
    'objective': 'binary:logistic',
    'eval_metric': 'error',
    'eta': 0.3,
    'nthread': 10,
}
dmatrix = xgb.DMatrix(X_train, label=y_train)
model = xgb.train(param, dmatrix, num_boost_round=200)

In [8]:
# 在测试集上用模型预测结果
dmatrix = xgb.DMatrix(X_test)
y_pred = model.predict(dmatrix)

In [9]:
# 测试集效果检验
from sklearn import metrics

auc_score = metrics.roc_auc_score(y_test, y_pred)          # 先计算AUC
y_pred = list(map(lambda x:1 if x > 0.5 else 0, y_pred))   # 二值化
print(metrics.classification_report(y_test, y_pred))
print("准确率:", metrics.accuracy_score(y_test, y_pred))
print("AUC:", auc_score)

              precision    recall  f1-score   support

           0       0.71      0.79      0.75       155
           1       0.90      0.86      0.88       345

    accuracy                           0.84       500
   macro avg       0.81      0.82      0.81       500
weighted avg       0.84      0.84      0.84       500

准确率: 0.836
AUC: 0.9067695184665732


## 使用已经训练好的模型进行检测


In [7]:
df_1 = pd.read_csv(r'C:\Users\20429\Desktop\国泰君安期货笔试\filtered_data_for_knn.csv')
df_1

,time,content,source
0,NaN,来源：公安部微信公众号\n 累计2876名缅甸妙瓦底地区的中国籍涉诈犯罪嫌疑人经泰国被...,中国基金报
1,NaN,厦门市海洋发展局网站，厦门市海洋发展局局长王宇近日在接受当地媒体采访时表示，“今年，我们要进...,每日经济新闻
2,NaN,新华财经北京3月14日电 3月14日，中国银行间市场交易商协会发布《银行间债券市场进一步支持...,新华财经
3,NaN,3月14日，智慧芽旗下智慧芽创新研究中心正式发布《新质生产力系列报告之2024年中国“专精特...,国际金融报
4,NaN,南都讯记者曾俊豪 3月13日，广东省人民政府办公厅印发《广东省促进银发经济高质量发展增进老年...,南方都市报
...,...,...,...
52298,NaN,近期以来，中国半导体设备产业在国家战略引导与市场需求驱动下加速突破。大基金二期通过注资昂坤视...,NaN
52299,NaN,向主要客户提早供应面向AI的超高性能DRAM新产品“12层HBM4”样品\n 经过验证后，...,NaN
52300,NaN,根据TrendForce集邦咨询最新AI Server供应链调查，预期NVIDIA（英伟达）...,NaN
52301,NaN,"根据TrendForce集邦咨询最新研究，2024年全球前十大IC设计业者营收合计约2,49...",NaN


In [10]:
new_X = vectorizer.transform(df_1.content.values.tolist())
new_metrics = xgb.DMatrix(new_X)
new_y_pred = model.predict(new_metrics)


In [13]:
y_pred = list(new_y_pred)  # 二值化
df_1['label'] = y_pred
df_1[df_1['label'] >0.8]


,time,content,source,label
365,NaN,1994年还是键盘输入的天下，彼时触摸屏并未出现，《连线》杂志创始主编凯文·凯利大胆预言：“...,每日经济新闻,0.847566
963,NaN,2025年中央一号文件对挖掘花生扩种潜力作出部署。这是中央一号文件首次针对花生产业提出发展意...,经济日报,0.851540
1200,NaN,上证报中国证券网讯（记者郭晓萍）记者从上海市经济和信息化委员会获悉，3月18日，为加快推进具...,上海证券报·中国证券网,0.838047
1480,NaN,3月14日，理想汽车2024年第四季度及全年财报出炉。继2023年之后，理想汽车营收继续突破...,时代财经,0.814819
1726,NaN,由“特朗普关税”挑起的贸易战，让美国本土企业特斯拉也感到了威胁！\n 近期，美国总统特...,券商中国,0.893335
...,...,...,...,...
50979,NaN,1994年还是键盘输入的天下，彼时触摸屏并未出现，《连线》杂志创始主编凯文·凯利大胆预言：“...,每日经济新闻,0.847566
51580,NaN,2025年中央一号文件对挖掘花生扩种潜力作出部署。这是中央一号文件首次针对花生产业提出发展意...,经济日报,0.851540
51817,NaN,上证报中国证券网讯（记者郭晓萍）记者从上海市经济和信息化委员会获悉，3月18日，为加快推进具...,上海证券报·中国证券网,0.838047
52238,NaN,AI搜索大战打到农村！腾讯元宝，盯上“母猪产后护理” \n 险资与能源化工产业应深度协同 ...,NaN,0.862105


In [ ]:
# 通过langchain选择置信水平搞的作为验证集
